In [64]:
%pip install --upgrade pip
%pip install yfinance torch numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [65]:
import yfinance as yf
import torch.nn as nn
import numpy as np
import torch 
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from datetime import datetime, timedelta

In [66]:
AAPL = yf.Ticker("AAPL")
AAPL.info

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download app

In [67]:
AAPL_hist = AAPL.history(period="1mo")
AAPL_hist.info

<bound method DataFrame.info of                                  Open        High         Low       Close  \
Date                                                                        
2026-04-22 00:00:00-04:00  267.573497  273.488031  266.624360  272.918579   
2026-04-23 00:00:00-04:00  274.796818  275.516157  271.399954  273.178314   
2026-04-24 00:00:00-04:00  272.508933  272.808645  269.401780  270.810486   
2026-04-27 00:00:00-04:00  265.845058  268.112957  264.826008  267.363647   
2026-04-28 00:00:00-04:00  272.089320  272.978515  268.412715  270.460815   
2026-04-29 00:00:00-04:00  267.303712  270.790520  266.794202  269.921326   
2026-04-30 00:00:00-04:00  270.251027  275.745964  267.893213  271.100250   
2026-05-01 00:00:00-04:00  278.603290  286.955610  278.113751  279.882141   
2026-05-04 00:00:00-04:00  279.402577  280.371685  274.606977  276.575165   
2026-05-05 00:00:00-04:00  276.675100  284.308082  276.245503  283.918427   
2026-05-06 00:00:00-04:00  281.660510  287.7

In [68]:
ticker = "AAPL"
Seq_len = 60 #number of months for the input window
Hidden_size = 128
num_layers = 2
dropout = .2
batch_size = 32
lr = 1e-3
epochs = 20
Use_log_returns = True
Device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"using: {Device}")


using: mps


In [69]:
def fetch_ohlcv(ticker: str, years: int = 5) -> np.array:
    end = datetime.today()
    start = end - timedelta(days = years*365)
    df = yf.download(ticker, start, end, auto_adjust = True, progress = False)
    df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
    return df.values.astype(np.float64), df.index.tolist()
raw, dates = fetch_ohlcv(ticker=ticker)
print(f"Fetched {len(raw)} trading days for {ticker}")

Fetched 1255 trading days for AAPL


In [70]:
def compute_features(raw: np.ndarray) ->np.ndarray:
    if Use_log_returns:
        price = raw[:,:4]
        volume = raw[:,:4:5]

        log_ret = np.log(price[1:] /  price[:-1])
        vol_chg = np.log(volume[1:] /  volume[:-1])

        close_ret = log_ret[:,3]
        rolling_std = np.array([
            close_ret[max(0, i-9): i+1].std()
            for i in range(len(close_ret))
        ]).reshape(-1, 1)

        features = np.hstack([log_ret, vol_chg, rolling_std])
    else:
        scaler = StandardScaler()
        features = scaler.fit.transform(raw)
        features = features[1:]
    return features.astype(np.float32)

features = compute_features(raw)
print(f"feature matrix shape: {features.shape} (timesteps, features)")


feature matrix shape: (1254, 6) (timesteps, features)


In [74]:
class priceSequenceDataset(Dataset):
    def __init__(self, features: np.ndarray, seq_len: int = Seq_len):
        self.features = torch.tensor(features)
        self.seq_len = seq_len
    
    def __len__(self):
        return len(self.features) - self.seq_len
    
    def __getitem__(self,idx):
        x = self.features[idx: idx + self.seq_len]
        y = self.features[idx + self.seq_len, 3]
        return x,y
    
dataset = priceSequenceDataset(features)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last = True)
print(f"Dataset: {len(dataset)} samples | Batches per epoch {len(loader)}")

Dataset: 1194 samples | Batches per epoch 37
